In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

print("All libraries imported!")

In [ ]:

train_df = pd.read_csv('/kaggle/input/titanic/train.csv')
test_df = pd.read_csv('/kaggle/input/titanic/test.csv')

test_passenger_id = test_df['PassengerId']

print("Data loaded successfully.")
print(train_df.head())

In [ ]:
print(train_df[['Sex', 'Survived']].groupby(['Sex']).mean())

In [ ]:
sns.barplot(x='Pclass', y='Survived', data=train_df)
plt.title('Survival Rate by Passenger Class')
plt.show()

In [ ]:
y_train = train_df['Survived']
train_df = train_df.drop('Survived', axis=1)

all_data = pd.concat([train_df, test_df], ignore_index=True)

# --- NEW FEATURE: Extract Title from Name ---
all_data['Title'] = all_data['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
rare_titles = ['Dr', 'Rev', 'Col', 'Major', 'Lady', 'Sir', 'Capt', 'Don', 'Jonkheer', 'Dona', 'Mlle', 'Countess', 'Ms', 'Mme']
all_data['Title'] = all_data['Title'].replace(rare_titles, 'Rare')
all_data['Title'] = all_data['Title'].fillna(0)
title_mapping = {"Mr": 1, "Miss": 2, "Mrs": 3, "Master": 4, "Rare": 5}
all_data['Title'] = all_data['Title'].map(title_mapping)
# --------------------------------------------

median_age = all_data['Age'].median()
all_data['Age'] = all_data['Age'].fillna(median_age)

mode_embarked = all_data['Embarked'].mode()[0]
all_data['Embarked'] = all_data['Embarked'].fillna(mode_embarked)

median_fare = all_data['Fare'].median()
all_data['Fare'] = all_data['Fare'].fillna(median_fare)

all_data['FamilySize'] = all_data['SibSp'] + all_data['Parch'] + 1
all_data['IsAlone'] = 0
all_data.loc[all_data['FamilySize'] == 1, 'IsAlone'] = 1

all_data['Sex'] = all_data['Sex'].map({'male': 0, 'female': 1})
all_data = pd.get_dummies(all_data, columns=['Embarked'], drop_first=True)

all_data = all_data.drop(['Cabin', 'Name', 'Ticket', 'SibSp', 'Parch', 'PassengerId'], axis=1)

print("Data is all clean and ready with the new 'Title' feature!")
print(all_data.head())

In [ ]:
X_train = all_data.iloc[:len(train_df)]
X_test = all_data.iloc[len(train_df):]

print("Data split back into X_train and X_test.")

In [ ]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 8, 10, 12],
    'min_samples_leaf': [1, 2, 4]
}

model = RandomForestClassifier(random_state=42)
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, 
                           cv=5, n_jobs=-1, verbose=1)

print("Starting Grid Search to find the best model...")
grid_search.fit(X_train, y_train)

print("\nGrid Search finished!")
print(f"Best Accuracy found: {grid_search.best_score_:.4f}")
print(f"Best Parameters found: {grid_search.best_params_}")

In [ ]:
print("Making predictions with the best tuned model...")

best_model = grid_search.best_estimator_
predictions = best_model.predict(X_test)

submission = pd.DataFrame({
    "PassengerId": test_passenger_id,
    "Survived": predictions
})

submission.to_csv('/kaggle/working/submission.csv', index=False)

print("'submission.csv' file created with the improved model!")